In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.metrics import (roc_auc_score, f1_score, recall_score, precision_score, 
                            accuracy_score, confusion_matrix, classification_report)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
import time
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [4]:
X_train = np.load("../data/processed/X_train.npy")
X_val = np.load("../data/processed/X_val.npy")
X_test = np.load("../data/processed/X_test.npy")
y_train = np.load("../data/processed/y_train.npy")
y_val = np.load("../data/processed/y_val.npy")
y_test = np.load("../data/processed/y_test.npy")

In [6]:
print("="*60)
print("ДАННЫЕ ЗАГРУЖЕНЫ")
print("="*60)
print(f"Train: {X_train.shape}, Target: {y_train.shape}")
print(f"Val:   {X_val.shape}, Target: {y_val.shape}")
print(f"Test:  {X_test.shape}, Target: {y_test.shape}")
print(f"Class balance in train: {np.sum(y_train==0)} legit, {np.sum(y_train==1)} fraud")

ДАННЫЕ ЗАГРУЖЕНЫ
Train: (398041, 29), Target: (398041,)
Val:   (85294, 29), Target: (85294,)
Test:  (85295, 29), Target: (85295,)
Class balance in train: 199020 legit, 199021 fraud


In [7]:
def evaluate_model(model, X_train, y_train, X_test, y_test, name="model", return_proba=False):
    """Обучает и оценивает модель на тестовой выборке"""
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        'name': name,
        'roc_auc': roc_auc_score(y_test, y_proba),
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'train_time': train_time
    }
    
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"Train time: {train_time:.2f} sec")
    print(f"ROC-AUC:    {metrics['roc_auc']:.6f}")
    print(f"Accuracy:   {metrics['accuracy']:.6f}")
    print(f"Precision:  {metrics['precision']:.6f}")
    print(f"Recall:     {metrics['recall']:.6f}")
    print(f"F1-Score:   {metrics['f1']:.6f}")
    
    if return_proba:
        return metrics, y_proba
    return metrics


In [9]:
baseline_results = []

In [10]:
lr_default = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
baseline_results.append(evaluate_model(lr_default, X_train, y_train, X_test, y_test, "Baseline_LR"))


Baseline_LR
Train time: 0.53 sec
ROC-AUC:    0.993440
Accuracy:   0.964347
Precision:  0.977250
Recall:     0.950829
F1-Score:   0.963858


In [11]:
knn_default = KNeighborsClassifier(n_neighbors=5)
baseline_results.append(evaluate_model(knn_default, X_train, y_train, X_test, y_test, "Baseline_KNN"))


Baseline_KNN
Train time: 0.03 sec
ROC-AUC:    0.999214
Accuracy:   0.997468
Precision:  0.994961
Recall:     1.000000
F1-Score:   0.997474


In [12]:
dt_default = DecisionTreeClassifier(random_state=RANDOM_STATE)
baseline_results.append(evaluate_model(dt_default, X_train, y_train, X_test, y_test, "Baseline_DT"))


Baseline_DT
Train time: 21.35 sec
ROC-AUC:    0.997491
Accuracy:   0.997491
Precision:  0.996397
Recall:     0.998593
F1-Score:   0.997494


In [13]:
rf_default = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
baseline_results.append(evaluate_model(rf_default, X_train, y_train, X_test, y_test, "Baseline_RF"))


Baseline_RF
Train time: 54.26 sec
ROC-AUC:    0.999987
Accuracy:   0.999742
Precision:  0.999484
Recall:     1.000000
F1-Score:   0.999742


In [16]:
gb_default = GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE)
baseline_results.append(evaluate_model(gb_default, X_train, y_train, X_test, y_test, "Baseline_GB"))


Baseline_GB
Train time: 325.78 sec
ROC-AUC:    0.998490
Accuracy:   0.979202
Precision:  0.988479
Recall:     0.969705
F1-Score:   0.979002


In [17]:
ada_default = AdaBoostClassifier(n_estimators=100, random_state=RANDOM_STATE)
baseline_results.append(evaluate_model(ada_default, X_train, y_train, X_test, y_test, "Baseline_AdaBoost"))


Baseline_AdaBoost
Train time: 127.02 sec
ROC-AUC:    0.994621
Accuracy:   0.963339
Precision:  0.975069
Recall:     0.950993
F1-Score:   0.962880


In [18]:
baseline_df = pd.DataFrame(baseline_results)
print("\n" + "="*80)
print("BASELINE MODELS COMPARISON (Default Parameters)")
print("="*80)
print(baseline_df[['name', 'roc_auc', 'f1', 'recall', 'train_time']].to_string(index=False, float_format='%.6f'))


BASELINE MODELS COMPARISON (Default Parameters)
             name  roc_auc       f1   recall  train_time
      Baseline_LR 0.993440 0.963858 0.950829    0.534693
     Baseline_KNN 0.999214 0.997474 1.000000    0.026378
      Baseline_DT 0.997491 0.997494 0.998593   21.352968
      Baseline_RF 0.999987 0.999742 1.000000   54.260046
      Baseline_GB 0.998490 0.979002 0.969705  325.779440
Baseline_AdaBoost 0.994621 0.962880 0.950993  127.020874


In [21]:
print("\n" + "="*30)
print("EXPERIMENT 1: Logistic Regression - Hyperparameter Tuning")
print("="*30)


EXPERIMENT 1: Logistic Regression - Hyperparameter Tuning


In [23]:
lr_params = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'liblinear', 'newton-cg'],
    'penalty': ['l2'],
    'max_iter': [1000, 2000]
}

In [24]:
lr_grid = GridSearchCV(
    LogisticRegression(random_state=RANDOM_STATE),
    lr_params,
    cv=cv_strategy,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)


In [32]:
start_time = time.time()
lr_grid.fit(X_train, y_train)
lr_tune_time = time.time() - start_time

best_lr = lr_grid.best_estimator_
y_pred = best_lr.predict(X_test)
y_proba = best_lr.predict_proba(X_test)[:, 1]

print(f"\nBest Logistic Regression Parameters:")
print(f"   {lr_grid.best_params_}")
print(f"   Best CV Score: {lr_grid.best_score_:.6f}")
print(f"   Tuning time: {lr_tune_time:.2f} sec")

lr_tuned_metrics = {
    'name': 'LR_Tuned',
    'roc_auc': roc_auc_score(y_test, y_proba),
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'train_time': lr_tune_time
}

print(f"\nTest Results:")
print(f"   ROC-AUC: {lr_tuned_metrics['roc_auc']:.6f}")
print(f"   F1: {lr_tuned_metrics['f1']:.6f}")


Fitting 5 folds for each of 36 candidates, totalling 180 fits

Best Logistic Regression Parameters:
   {'C': 100, 'max_iter': 1000, 'penalty': 'l2', 'solver': 'newton-cg'}
   Best CV Score: 0.993536
   Tuning time: 51.24 sec

Test Results:
   ROC-AUC: 0.993454
   F1: 0.964026


In [33]:
all_results = baseline_results + [lr_tuned_metrics]
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('roc_auc', ascending=False)

In [34]:
print(results_df[['name', 'roc_auc', 'f1', 'recall', 'precision', 'train_time']].to_string(
    index=False, float_format='%.6f'))

             name  roc_auc       f1   recall  precision  train_time
      Baseline_RF 0.999987 0.999742 1.000000   0.999484   54.260046
     Baseline_KNN 0.999214 0.997474 1.000000   0.994961    0.026378
      Baseline_GB 0.998490 0.979002 0.969705   0.988479  325.779440
      Baseline_DT 0.997491 0.997494 0.998593   0.996397   21.352968
Baseline_AdaBoost 0.994621 0.962880 0.950993   0.975069  127.020874
         LR_Tuned 0.993454 0.964026 0.951016   0.977395   51.237551
      Baseline_LR 0.993440 0.963858 0.950829   0.977250    0.534693
